# QuantBench 模型矩阵外部知识验证

本 notebook 只验证 `arxiv:2504.18600` 作为 QuantBench 模型矩阵参考来源是否已经具备 research 门禁证据；不复制论文全文，不 vendoring 外部代码，也不把外部来源直接写入生产逻辑。

In [1]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

import pandas as pd


def locate_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "AGENTS.md").exists() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError("cannot locate repo root from notebook cwd")


REPO_ROOT = locate_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

CARD_DIR = REPO_ROOT / "research" / "quantbench_model_matrix_validation_v1"
SOURCE_CARD = REPO_ROOT / "research" / "quantbench_all_model_run_ledger_v1"
CARD_DIR.mkdir(parents=True, exist_ok=True)

from src.core.knowledge import KnowledgeSource, build_knowledge_backlog, build_knowledge_research_evidence_index
from src.model.registry import QUANTBENCH_MODEL_NAMES, REQUESTED_QUANTBENCH_MODEL_NAMES


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


## 来源和运行证据

QuantBench 来源只保留 arXiv URL、自有摘要和结构化假设。模型训练、预测、回测和分析证据来自本仓库已经执行完成的 `research/quantbench_all_model_run_ledger_v1`。

In [2]:
source = KnowledgeSource(
    source_id="arxiv:2504.18600",
    source_type="arxiv",
    title="QuantBench: Benchmarking AI Methods for Quantitative Investment",
    url="https://arxiv.org/abs/2504.18600",
    summary=(
        "Benchmark reference for a standardized full-pipeline quant workflow, model matrix coverage, "
        "continual-learning stress, relation modeling, and overfitting control."
    ),
    license_note="arXiv metadata and self-written summary only.",
    retrieved_at="2026-06-19",
    access_status="metadata_accessible",
)
backlog = build_knowledge_backlog([source])
backlog_item = backlog[0]

ledger_summary = read_json(SOURCE_CARD / "model_run_ledger_summary.json")
analysis_summary = read_json(SOURCE_CARD / "analysis_summary.json")
analysis_manifest = read_json(SOURCE_CARD / "analysis_manifest.json")
notebook_summary = read_json(SOURCE_CARD / "notebook_summary.json")
ledger = pd.read_csv(SOURCE_CARD / "model_run_ledger.csv")
capabilities = pd.read_csv(SOURCE_CARD / "model_capabilities.csv")

requested_models = list(REQUESTED_QUANTBENCH_MODEL_NAMES)
registered_models = list(QUANTBENCH_MODEL_NAMES)
ledger_models = ledger["model"].astype(str).tolist()

assert requested_models == registered_models
assert set(ledger_models) == set(requested_models)
assert ledger_summary["expected_model_count"] == len(requested_models) == 32
assert ledger_summary["all_models_completed"] is True
assert ledger_summary["proxy_model_count"] == 0
assert ledger_summary["adapter_statuses"] == ["paper_strict_adapter"]


## 覆盖检查

覆盖检查以仓库注册表为准，确认 32 个用户点名模型均完成训练、预测、回测、objective、promotion 和 analysis 记录。

In [3]:
status_columns = [
    "registered",
    "capability_recorded",
    "trained",
    "predicted",
    "backtested",
    "objective_evaluated",
    "promotion_evaluated",
    "analysis_recorded",
    "manifested",
]
completion_counts = {column: int(ledger[column].astype(bool).sum()) for column in status_columns}
assert all(count == 32 for count in completion_counts.values())

family_coverage = (
    capabilities.groupby(["family", "objective", "input_kind", "paper_fidelity"], dropna=False)
    .size()
    .reset_index(name="model_count")
    .sort_values(["family", "objective", "input_kind", "paper_fidelity"])
)
family_coverage.to_csv(CARD_DIR / "model_family_coverage.csv", index=False)

license_review = "\n".join(
    [
        "# QuantBench 来源许可 Review",
        "",
        "## 中文结论",
        "",
        "- 来源：arXiv `2504.18600`，标题为 `QuantBench: Benchmarking AI Methods for Quantitative Investment`。",
        "- 使用方式：只保存 URL、自有摘要、结构化假设和本仓库运行证据索引。",
        "- 未复制论文全文，未复制论文图表，未 vendoring 外部代码。",
        "- 是否晋升：否。本卡只解除外部知识 backlog 的 research 证据缺口；是否进入生产仍要走独立 production review。",
        "- 风险：论文方法可能与本仓库 compact adapter、SOXL 单标的样本和短窗口设置存在差异，不能把 benchmark 结论直接当成交易结论。",
    ]
) + "\n"
(CARD_DIR / "license_review.md").write_text(license_review, encoding="utf-8")

shutil.copyfile(SOURCE_CARD / "model_run_ledger_summary.json", CARD_DIR / "model_run_ledger_summary.json")

model_matrix_evidence = {
    "schema_version": "research.quantbench_model_matrix_evidence.v1",
    "source_run_card": "research/quantbench_all_model_run_ledger_v1",
    "source_notebook": "research/quantbench_all_model_run_ledger_v1/quantbench_all_model_run_ledger_v1.ipynb",
    "analysis_report_schema_version": analysis_manifest["metadata"]["analysis_report_schema_version"],
    "model_run_ledger_schema_version": ledger_summary["schema_version"],
    "analysis_summary_schema_version": analysis_summary.get("schema_version", "analysis.report.v1"),
    "requested_model_count": len(requested_models),
    "registered_model_count": len(registered_models),
    "trained_model_count": int(ledger_summary["trained_model_count"]),
    "backtested_model_count": int(ledger_summary["backtested_model_count"]),
    "all_models_completed": bool(ledger_summary["all_models_completed"]),
    "adapter_statuses": list(ledger_summary["adapter_statuses"]),
    "proxy_model_count": int(ledger_summary["proxy_model_count"]),
    "paper_fidelity_counts": dict(ledger_summary["paper_fidelity_counts"]),
    "top_model": notebook_summary.get("top_model", ""),
    "top_return_pct": notebook_summary.get("top_return_pct", 0),
    "artifact_paths": {
        "model_run_ledger": "research/quantbench_all_model_run_ledger_v1/model_run_ledger.csv",
        "model_capabilities": "research/quantbench_all_model_run_ledger_v1/model_capabilities.csv",
        "analysis_manifest": "research/quantbench_all_model_run_ledger_v1/analysis_manifest.json",
        "analysis_summary": "research/quantbench_all_model_run_ledger_v1/analysis_summary.json",
        "model_family_coverage": "research/quantbench_model_matrix_validation_v1/model_family_coverage.csv",
    },
}
write_json(CARD_DIR / "model_matrix_evidence.json", model_matrix_evidence)

source_provenance = {
    "schema_version": "knowledge.source_provenance.v1",
    "source_id": source.source_id,
    "source_type": source.source_type,
    "title": source.title,
    "url": source.url,
    "retrieved_at": source.retrieved_at,
    "access_status": source.access_status,
    "source_used_as": "hypothesis_only",
    "usage_scope": "hypothesis_and_adapter_reference",
    "source_reliability": "primary_metadata",
    "license_note": "只保存 arXiv URL、自有摘要和结构化假设；未复制论文全文。",
    "external_source_checks": {
        "arxiv_url": source.url,
        "copied_paper_text": False,
        "copied_paper_full_text": False,
        "copied_paper_figures": False,
        "vendored_external_code": False,
        "implementation_source": "repo_owned_quantbench_all_model_run_ledger_v1",
    },
}
write_json(CARD_DIR / "source_provenance.json", source_provenance)


In [4]:
evidence_index = build_knowledge_research_evidence_index(REPO_ROOT, backlog)
evidence_requirements = evidence_index["quantbench_model_matrix"]

summary = {
    "schema_version": "research.quantbench_model_matrix_validation.v1",
    "source_id": source.source_id,
    "candidate_name": backlog_item["candidate_name"],
    "target_subsystem": backlog_item["target_subsystem"],
    "target_interface": backlog_item["target_interface"],
    "requested_model_count": len(requested_models),
    "registered_model_count": len(registered_models),
    "ledger_model_count": int(ledger_summary["ledger_model_count"]),
    "trained_model_count": int(ledger_summary["trained_model_count"]),
    "predicted_model_count": int(ledger_summary["predicted_model_count"]),
    "backtested_model_count": int(ledger_summary["backtested_model_count"]),
    "objective_evaluated_model_count": int(ledger_summary["objective_evaluated_model_count"]),
    "promotion_evaluated_model_count": int(ledger_summary["promotion_evaluated_model_count"]),
    "analysis_recorded_model_count": int(ledger_summary["analysis_recorded_model_count"]),
    "manifested_model_count": int(ledger_summary["manifested_model_count"]),
    "all_models_completed": bool(ledger_summary["all_models_completed"]),
    "adapter_statuses": list(ledger_summary["adapter_statuses"]),
    "proxy_model_count": int(ledger_summary["proxy_model_count"]),
    "native_model_count": int(ledger_summary["native_model_count"]),
    "paper_fidelity_counts": dict(ledger_summary["paper_fidelity_counts"]),
    "model_families": sorted(capabilities["family"].astype(str).unique().tolist()),
    "objective_types": sorted(capabilities["objective"].astype(str).unique().tolist()),
    "completion_counts": completion_counts,
    "implementation_allowed": False,
    "promotion_target": "research_only",
    "evidence_requirements": evidence_requirements,
    "source_run_card": "research/quantbench_all_model_run_ledger_v1",
    "source_used_as": "hypothesis_only",
}
write_json(CARD_DIR / "quantbench_model_matrix_summary.json", summary)
summary


{'schema_version': 'research.quantbench_model_matrix_validation.v1',
 'source_id': 'arxiv:2504.18600',
 'candidate_name': 'quantbench_model_matrix',
 'target_subsystem': 'src/model',
 'target_interface': 'src/model.registry.ModelSpec + src.model.training.fit_predict_model',
 'requested_model_count': 32,
 'registered_model_count': 32,
 'ledger_model_count': 32,
 'trained_model_count': 32,
 'predicted_model_count': 32,
 'backtested_model_count': 32,
 'objective_evaluated_model_count': 32,
 'promotion_evaluated_model_count': 32,
 'analysis_recorded_model_count': 32,
 'manifested_model_count': 32,
 'all_models_completed': True,
 'adapter_statuses': ['paper_strict_adapter'],
 'proxy_model_count': 0,
 'native_model_count': 32,
 'paper_fidelity_counts': {'native_graph': 9,
  'native_library': 10,
  'native_sequence': 7,
  'native_torch': 6},
 'model_families': ['boosting',
  'graph',
  'linear',
  'neural_tabular',
  'sequence',
  'tree'],
 'objective_types': ['classification', 'ic', 'mse', '

## 中文结论

- 做了什么：把 `arxiv:2504.18600` QuantBench 来源转成 `quantbench_model_matrix` research 验证卡；只保存 URL、自有摘要和结构化假设，并复用本仓库 `quantbench_all_model_run_ledger_v1` 的训练、回测、objective、promotion、analysis 证据。
- 核心指标：32 个用户点名模型均已注册、训练、预测、回测、objective 评估、promotion 评估、analysis 记录和 manifest 记录；adapter_statuses 为 `paper_strict_adapter`；proxy_model_count 为 0；paper_fidelity 覆盖 `native_library`、`native_sequence`、`native_torch`、`native_graph`。
- regime 差异：本卡不重新计算 regime；regime/window/model difference 证据来自源运行卡的 `analysis_summary.json`、`regime_metrics.csv`、`window_metrics.csv` 和 `model_regime_window_diagnostics.csv`。
- 是否晋升：否。本卡只证明外部知识 backlog 的 research 门禁已经满足，允许后续进入 `src/model` 评审；不代表任何模型或生产策略晋升。
- 风险：QuantBench 论文是 benchmark 参考，本仓库当前是 SOXL 单标的短窗口 compact adapter 证据，不能把论文结论或本次 smoke 结果直接外推为实盘优势。
- 下一步：继续保留 production readiness review gate，生产晋升必须基于独立 production notebook、长窗口 replay、成本压力和 paper API 守卫验证。